# 01 — Parse

Builds **LIB**, **CORPUS**, and **VOCAB** tables from the raw Studio Ghibli screenplay `.txt` files.

**OHCO**: `film_name → chunk_num → sent_num → token_num`  
A *chunk* is a single speaker utterance. Only dialogue with speaker attribution is retained;
stage directions, action lines, and narration are dropped to ensure cross-film comparability
(the 14 files range from rich full-screenplay format to bare SRT subtitles with no stage directions).

`speaker` is stored as a regular CORPUS column (not an OHCO level) because SRT-format
films carry no per-character attribution.

## Setup

In [13]:
import pandas as pd
import numpy as np
import re
import os
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)

DATA_DIR = 'kaggle-dataset'
OUTPUT_DIR = 'tables'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Film Metadata

- `start_line`: 0-indexed line where actual screenplay content begins (skips each file's header/preamble)
- `fmt`: which speaker-extraction strategy to apply
  - `dialogue` — chunk-based LISTSERV/fan-script format (most files)
  - `screenplay` — Hollywood format (all-caps character headers, prose action lines)
  - `timestamp` — subtitle-derived `[MM:SS]  SPEAKER: text` per line
  - `howls` — `Name    text` per line with indented continuation chunks
  - `srt` — raw SRT subtitles, no per-character attribution

In [14]:
OHCO = ['film_name', 'chunk_num', 'sent_num', 'token_num']

FILM_META = {
    'miyazaki_castle_in_the_sky':      {'director': 'Miyazaki', 'year': 1986, 'start_line': 93,  'fmt': 'dialogue'},
    'miyazaki_howls_moving_castle':    {'director': 'Miyazaki', 'year': 2004, 'start_line': 13,  'fmt': 'howls'},
    'miyazaki_kikis_delivery_service': {'director': 'Miyazaki', 'year': 1989, 'start_line': 39,  'fmt': 'dialogue'},
    'miyazaki_my_neighbor_totoro':     {'director': 'Miyazaki', 'year': 1988, 'start_line': 59,  'fmt': 'dialogue'},
    'miyazaki_nausicaa':               {'director': 'Miyazaki', 'year': 1984, 'start_line': 44,  'fmt': 'dialogue'},
    'miyazaki_ponyo':                  {'director': 'Miyazaki', 'year': 2008, 'start_line': 28,  'fmt': 'timestamp'},
    'miyazaki_porco_rosso':            {'director': 'Miyazaki', 'year': 1992, 'start_line': 27,  'fmt': 'dialogue'},
    'miyazaki_princess_mononoke':      {'director': 'Miyazaki', 'year': 1997, 'start_line': 19,  'fmt': 'dialogue'},
    'miyazaki_spirited_away':          {'director': 'Miyazaki', 'year': 2001, 'start_line': 6,   'fmt': 'screenplay'},
    'miyazaki_the_wind_rises':         {'director': 'Miyazaki', 'year': 2013, 'start_line': 29,  'fmt': 'timestamp'},
    'takahata_grave_of_the_fireflies': {'director': 'Takahata', 'year': 1988, 'start_line': 132, 'fmt': 'dialogue'},
    'takahata_only_yesterday':         {'director': 'Takahata', 'year': 1991, 'start_line': 118, 'fmt': 'dialogue'},
    'takahata_pom_poko':               {'director': 'Takahata', 'year': 1994, 'start_line': 0,   'fmt': 'srt'},
    'takahata_princess_kaguya':        {'director': 'Takahata', 'year': 2013, 'start_line': 0,   'fmt': 'srt'},
}

## Universal Noise Removal

Strip translator footnotes and music symbols from raw text before format-specific parsing.

In [15]:
def remove_noise(text):
    """Strip universal noise that appears in multiple files."""
    # {translator footnote blocks} — may span multiple lines (only_yesterday)
    text = re.sub(r'\{[^}]*\}', '', text, flags=re.DOTALL)
    # [ALT/NOTE/LT/ET: ...] inline translator annotations (grave_of_the_fireflies)
    text = re.sub(r'\[(?:ALT|NOTE|LT|ET)[^\]]*\]', '', text, flags=re.IGNORECASE)
    # <hidden reference> angle-bracket annotations (only_yesterday)
    text = re.sub(r'<[^>]{1,60}>', '', text)
    # Music note symbols
    text = re.sub(r'[\u266a\u266b]', '', text)
    return text

## Format-Specific Utterance Extractors

Each extractor returns a list of `(speaker, text)` tuples representing speaker-attributed dialogue chunks.
Stage directions, action lines, and unattributed narration are dropped.

**Key design note**: `extract_dialogue` and `extract_screenplay` use **line-by-line state tracking**
rather than chunk splitting. Chunk splitting failed for two reasons:
- Many files (totoro, nausicaa, kiki) have `Speaker: dialogue` lines with **no blank line between them**,
  so consecutive speakers collapse into one chunk attributed to the first speaker
- Some chunks start with a stage direction `(...)` followed immediately by dialogue lines —
  dropping the whole chunk also drops all the dialogue inside it
- spirited_away has **no blank lines** between screenplay elements at all

In [16]:
# ---------------------------------------------------------------------------
# SRT format  (pom_poko, princess_kaguya)
# ---------------------------------------------------------------------------
def extract_srt(text):
    """SRT has no per-character labels. Italic subtitles → NARRATOR; plain → UNKNOWN."""
    text = re.sub(r'^\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^\d{2}:\d{2}:\d{2},\d+ --> \d{2}:\d{2}:\d{2},\d+\s*$', '',
                  text, flags=re.MULTILINE)

    utterances = []
    for block in re.split(r'\n\s*\n', text):
        block = block.strip()
        if not block:
            continue
        content_lines = [l.strip() for l in block.split('\n') if l.strip()]
        all_italic = all(re.match(r'^<i>', l) for l in content_lines)
        speaker = 'NARRATOR' if all_italic else 'UNKNOWN'
        clean = re.sub(r'<[^>]+>', '', block).strip()
        if clean:
            utterances.append((speaker, clean))
    return utterances


# ---------------------------------------------------------------------------
# Timestamp format  (ponyo, wind_rises)
# Format: [MM:SS]  SPEAKER: dialogue text
# ---------------------------------------------------------------------------
def extract_timestamp(text):
    """Each line is one utterance: [MM:SS]  SPEAKER: text."""
    text = re.sub(r'^─{3,}.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^\s*\[scene break[^\]]*\]\s*$', '', text,
                  flags=re.MULTILINE | re.IGNORECASE)

    utterances = []
    for line in text.split('\n'):
        line = line.strip()
        line = re.sub(r'^\[\d+:\d+\]\s+', '', line)
        m = re.match(r'^([A-Z][A-Z_]+):\s+(.+)$', line)
        if m:
            utterances.append((m.group(1), m.group(2)))
    return utterances


# ---------------------------------------------------------------------------
# Howl's Moving Castle format
# Lines: "Name    dialogue"  OR  "            continuation" (many leading spaces)
# ---------------------------------------------------------------------------
def extract_howls(text):
    """Line-by-line extraction; continuation lines (4+ leading spaces) go to last speaker."""
    text = re.sub(r'^\[(?:Scene|Action):.*?\]\s*$', '', text,
                  flags=re.MULTILINE | re.IGNORECASE | re.DOTALL)

    utterances = []
    last_speaker = None
    for line in text.split('\n'):
        m = re.match(r'^([A-Za-z][A-Za-z\s]+?)\s{4,}(.+)$', line)
        if m:
            last_speaker = m.group(1).strip()
            utterances.append((last_speaker, m.group(2).strip()))
        elif line.strip() and re.match(r'^\s{4,}', line) and last_speaker:
            utterances.append((last_speaker, line.strip()))
    return utterances


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def is_character_name(line):
    """Return True if line looks like a screenplay character name (short, no sentence punctuation)."""
    bare = re.sub(r'\s*\([^)]+\)\s*', '', line).strip()
    if not bare or len(bare) > 50:
        return False
    if re.search(r'[.!?,]', bare):
        return False
    if re.match(r"^[A-Z][A-Z\s'/\-\.]+$", bare):
        return True
    if re.match(r'^[A-Z][a-z]+(?:\s+[A-Za-z]+){0,3}$', bare):
        return True
    return False


# ---------------------------------------------------------------------------
# Dialogue format  (kiki, totoro, nausicaa, castle, porco, grave, only_yesterday, mononoke)
#
# Line-by-line state machine: avoids losing dialogue that follows a stage-direction
# line within the same blank-line block, and handles consecutive "Speaker: text"
# lines with no blank separator between them.
# ---------------------------------------------------------------------------
def extract_dialogue(text):
    """Line-by-line speaker detection; flush on new speaker or blank line."""
    text = re.sub(r'^[\-=+*~_]{3,}\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^\s*\d+\.\s*$', '', text, flags=re.MULTILINE)

    utterances = []
    cur_speaker = None
    cur_lines   = []

    def flush():
        nonlocal cur_speaker, cur_lines
        if cur_speaker and cur_lines:
            utterances.append((cur_speaker, ' '.join(cur_lines)))
        cur_speaker = None
        cur_lines   = []

    PAT_A = re.compile(r"^([A-Za-z][A-Za-z\s'/\-\.]*?)(?:\s*\([^)]+\))?:\s+(.+)$")
    PAT_B = re.compile(r"^([A-Z][A-Z\s'/\-]+):$")
    PAT_D = re.compile(r"^([A-Z][A-Z\s'/\-/]+)\s{4,}(.+)$")

    for line in text.split('\n'):
        stripped = line.strip()

        if not stripped:
            flush()
            continue

        # Stage direction / action annotation → skip line, keep current speaker
        if stripped.startswith('(') or stripped.startswith('['):
            continue

        # Pattern A: "SPEAKER: dialogue"  (optional parenthetical before colon)
        m = PAT_A.match(stripped)
        if m:
            flush()
            cur_speaker = m.group(1).strip()
            cur_lines   = [m.group(2).strip()]
            continue

        # Pattern D: "SPEAKER    dialogue"  (4+ spaces, no colon) — grave
        m = PAT_D.match(stripped)
        if m:
            flush()
            cur_speaker = m.group(1).strip()
            cur_lines   = [m.group(2).strip()]
            continue

        # Pattern B: "SPEAKER:" alone — mononoke
        m = PAT_B.match(stripped)
        if m:
            flush()
            cur_speaker = m.group(1).strip()
            cur_lines   = []
            continue

        # Pattern C: character name alone on a line — castle, porco, only_yesterday
        if is_character_name(stripped):
            flush()
            cur_speaker = stripped
            cur_lines   = []
            continue

        # Continuation: append to current speaker
        if cur_speaker is not None:
            cur_lines.append(stripped)
        # else: unattributed narration/action → DROP

    flush()
    return utterances


# ---------------------------------------------------------------------------
# Screenplay format  (spirited_away)
#
# Line-by-line state machine with two states:
#   LOOKING   — waiting for a character-name line
#   COLLECTING — gathering dialogue lines under the current speaker
# Scene headings (INT./EXT./FADE/CUT) and clear action lines flush+reset.
# ---------------------------------------------------------------------------
def extract_screenplay(text):
    """Line-by-line screenplay extraction using LOOKING / COLLECTING states."""
    text = re.sub(r'^\s*\d+\.\s*$', '', text, flags=re.MULTILINE)

    utterances  = []
    cur_speaker = None
    cur_lines   = []

    def flush():
        nonlocal cur_speaker, cur_lines
        if cur_speaker and cur_lines:
            utterances.append((cur_speaker, ' '.join(cur_lines)))
        cur_speaker = None
        cur_lines   = []

    SCENE_HDR   = re.compile(r'^(?:INT\.|EXT\.|FADE|CUT|DISSOLVE|SMASH|TITLE|BLACK)', re.IGNORECASE)
    ACTION_COMMA = re.compile(r'^[A-Z]{2,}[A-Z\s]*,')   # e.g. "CHIHIRO, a ten-year-old"
    CHAR_NAME   = re.compile(r"^[A-Z][A-Z\s'\/\-]+(?:\s*\([\w\s'\.\/]+\))?$")

    for line in text.split('\n'):
        stripped = line.strip()

        if not stripped:
            flush()
            continue

        # Scene headings → flush + drop
        if SCENE_HDR.match(stripped):
            flush()
            continue

        # Parenthetical stage direction on its own line → skip, keep collecting
        if stripped.startswith('(') and stripped.endswith(')'):
            continue

        # Action line: ALL-CAPS prefix + comma + lowercase prose → flush + drop
        if ACTION_COMMA.match(stripped) and re.search(r'[a-z]', stripped):
            flush()
            continue

        # Character name: all-caps, no lowercase or commas, short
        bare = re.sub(r'\s*\([^)]+\)\s*', '', stripped).strip()
        if (CHAR_NAME.match(stripped)
                and not re.search(r'[a-z,]', stripped)
                and len(bare) <= 40):
            flush()
            cur_speaker = bare
            cur_lines   = []
            continue

        # In LOOKING state with no match → action/description → DROP
        if cur_speaker is None:
            continue

        # In COLLECTING state → dialogue line
        cur_lines.append(stripped)

    flush()
    return utterances

## Parse Function

Dispatches to the right extractor, then sentence-tokenizes and POS-tags each utterance.

In [17]:
stemmer = PorterStemmer()

def coarse_pos(tag):
    """Map Penn Treebank tag to coarse group: n/v/j/r/x."""
    if tag.startswith('NN'): return 'n'
    if tag.startswith('VB'): return 'v'
    if tag.startswith('JJ'): return 'j'
    if tag.startswith('RB'): return 'r'
    return 'x'

EXTRACTORS = {
    'srt':        extract_srt,
    'timestamp':  extract_timestamp,
    'howls':      extract_howls,
    'dialogue':   extract_dialogue,
    'screenplay': extract_screenplay,
}

def parse_film(film_name, meta):
    with open(f"{DATA_DIR}/{film_name}.txt", encoding='utf-8', errors='replace') as f:
        lines = f.readlines()

    text = ''.join(lines[meta['start_line']:])
    text = remove_noise(text)

    utterances = EXTRACTORS[meta['fmt']](text)

    rows = []
    for chunk_num, (speaker, dialogue) in enumerate(utterances):
        for sent_num, sent in enumerate(sent_tokenize(dialogue)):
            token_num = 0
            for token, tag in pos_tag(word_tokenize(sent)):
                term = re.sub(r'[\W_]+', '', token).lower()
                if not term:                  # skip pure-punctuation tokens
                    continue
                rows.append((
                    film_name, chunk_num, sent_num, token_num,
                    token, term, speaker, tag, coarse_pos(tag)
                ))
                token_num += 1

    return rows

## Build CORPUS

In [18]:
all_rows = []
for film_name, meta in FILM_META.items():
    rows = parse_film(film_name, meta)
    all_rows.extend(rows)
    print(f"{film_name:<45} {len(rows):>7,} tokens")

CORPUS = pd.DataFrame(
    all_rows,
    columns=OHCO + ['token_str', 'term_str', 'speaker', 'pos', 'pos_group']
).set_index(OHCO)

print(f"\nTotal CORPUS tokens: {len(CORPUS):,}")
CORPUS.head(10)

miyazaki_castle_in_the_sky                      6,591 tokens
miyazaki_howls_moving_castle                    8,673 tokens
miyazaki_kikis_delivery_service                 6,971 tokens
miyazaki_my_neighbor_totoro                     3,794 tokens
miyazaki_nausicaa                               7,165 tokens
miyazaki_ponyo                                  3,657 tokens
miyazaki_porco_rosso                            6,804 tokens
miyazaki_princess_mononoke                      8,429 tokens
miyazaki_spirited_away                         10,144 tokens
miyazaki_the_wind_rises                         6,006 tokens
takahata_grave_of_the_fireflies                 6,069 tokens
takahata_only_yesterday                         7,317 tokens
takahata_pom_poko                              12,523 tokens
takahata_princess_kaguya                        6,363 tokens

Total CORPUS tokens: 100,506


token_str term_str  \
film_name                  chunk_num sent_num token_num                      
miyazaki_castle_in_the_sky 0         0        0                Ah       ah   
                           1         0        0               Wah      wah   
                                     1        0                It       it   
                                              1                's        s   
                                              2                 a        a   
                                              3               gas      gas   
                                              4              bomb     bomb   
                                     2        0                It       it   
                                              1                's        s   
                                              2                an       an   

                                                         speaker  pos  \
film_name                  chunk_num sent_num token_num                 
miyazaki_castle_in_the_sky 0         0        0              MEN   NN   
                           1         0        0          CREWMAN   NN   
                                     1        0          CREWMAN  PRP   
                                              1          CREWMAN  VBZ   
                                              2          CREWMAN   DT   
                                              3          CREWMAN   NN   
                                              4          CREWMAN   NN   
                                     2        0          CREWMAN  PRP   
                                              1          CREWMAN  VBZ   
                                              2          CREWMAN   DT   

                                                        pos_group  
film_name                  chunk_num sent_num token_num            
miyazaki_castle_in_the_sky 0         0        0                 n  
                           1         0        0                 n  
                                     1        0                 x  
                                              1                 v  
                                              2                 x  
                                              3                 n  
                                              4                 n  
                                     2        0                 x  
                                              1                 v  
                                              2                 x

## Build LIB

In [19]:
lib_rows = []
for film_name, meta in FILM_META.items():
    film_tokens = CORPUS.xs(film_name, level='film_name')
    lib_rows.append({
        'film_name': film_name,
        'director':  meta['director'],
        'year':      meta['year'],
        'n_chunks':  film_tokens.index.get_level_values('chunk_num').nunique(),
        'n_tokens':  len(film_tokens),
    })

LIB = pd.DataFrame(lib_rows).set_index('film_name')
LIB

,director,year,n_chunks,n_tokens
film_name,,,,
miyazaki_castle_in_the_sky,Miyazaki,1986,907,6591
miyazaki_howls_moving_castle,Miyazaki,2004,1353,8673
miyazaki_kikis_delivery_service,Miyazaki,1989,692,6971
miyazaki_my_neighbor_totoro,Miyazaki,1988,489,3794
miyazaki_nausicaa,Miyazaki,1984,848,7165
miyazaki_ponyo,Miyazaki,2008,817,3657
miyazaki_porco_rosso,Miyazaki,1992,665,6804
miyazaki_princess_mononoke,Miyazaki,1997,760,8429
miyazaki_spirited_away,Miyazaki,2001,451,10144


## Build VOCAB

In [20]:
stop_words = set(stopwords.words('english'))

VOCAB = CORPUS['term_str'].value_counts().to_frame('n')
VOCAB.index.name = 'term_str'
VOCAB['p']           = VOCAB['n'] / VOCAB['n'].sum()
VOCAB['i']           = np.log2(1 / VOCAB['p'])
VOCAB['n_chars']     = VOCAB.index.str.len()
VOCAB['porter_stem'] = VOCAB.index.map(stemmer.stem)
VOCAB['stop']        = VOCAB.index.isin(stop_words)

# Most frequent POS tag per term
max_pos_idx = (
    CORPUS.groupby(['term_str', 'pos'])
    .size()
    .groupby(level='term_str')
    .idxmax()
    .map(lambda x: x[1])          # extract the pos from the (term, pos) tuple
)
VOCAB['max_pos']       = max_pos_idx
VOCAB['max_pos_group'] = VOCAB['max_pos'].map(coarse_pos)

VOCAB = VOCAB.sort_index()
print(f"VOCAB size: {len(VOCAB):,} unique terms")
VOCAB.head(10)

VOCAB size: 7,367 unique terms


,n,p,i,n_chars,porter_stem,stop,max_pos,max_pos_group
term_str,,,,,,,,
0,1,0.00001,16.616922,1,0,False,CD,x
000411451,1,0.00001,16.616922,9,000411451,False,CD,x
000548106,1,0.00001,16.616922,9,000548106,False,CD,x
000736,1,0.00001,16.616922,6,000736,False,CD,x
000739,1,0.00001,16.616922,6,000739,False,CD,x
000809949,1,0.00001,16.616922,9,000809949,False,CD,x
001021162,1,0.00001,16.616922,9,001021162,False,CD,x
001133843,1,0.00001,16.616922,9,001133843,False,CD,x
001728981,1,0.00001,16.616922,9,001728981,False,CD,x


## Sanity Checks

In [21]:
print("=== Tokens per film ===")
print(LIB[['director', 'year', 'n_chunks', 'n_tokens']].to_string())

print("\n=== Tokens by director ===")
print(LIB.groupby('director')['n_tokens'].agg(['sum', 'mean']).astype(int))

print("\n=== Top 20 terms by frequency ===")
print(VOCAB.sort_values('n', ascending=False).head(20)[['n', 'p', 'stop']])

=== Tokens per film ===
                                 director  year  n_chunks  n_tokens
film_name                                                          
miyazaki_castle_in_the_sky       Miyazaki  1986       907      6591
miyazaki_howls_moving_castle     Miyazaki  2004      1353      8673
miyazaki_kikis_delivery_service  Miyazaki  1989       692      6971
miyazaki_my_neighbor_totoro      Miyazaki  1988       489      3794
miyazaki_nausicaa                Miyazaki  1984       848      7165
miyazaki_ponyo                   Miyazaki  2008       817      3657
miyazaki_porco_rosso             Miyazaki  1992       665      6804
miyazaki_princess_mononoke       Miyazaki  1997       760      8429
miyazaki_spirited_away           Miyazaki  2001       451     10144
miyazaki_the_wind_rises          Miyazaki  2013       976      6006
takahata_grave_of_the_fireflies  Takahata  1988       549      6069
takahata_only_yesterday          Takahata  1991       833      7317
takahata_pom_poko       

In [22]:
# Spot-check: verify start_line and speaker extraction worked correctly for each film
for film_name in FILM_META:
    sample = CORPUS.xs(film_name, level='film_name').head(3)
    speakers = sample['speaker'].unique()
    first_tokens = ' '.join(sample['token_str'].tolist())
    print(f"{film_name:<45} speakers={speakers}  first_tokens='{first_tokens}'")

miyazaki_castle_in_the_sky                    speakers=['MEN' 'CREWMAN']  first_tokens='Ah Wah It'
miyazaki_howls_moving_castle                  speakers=['Bessie']  first_tokens='Sophie We just'
miyazaki_kikis_delivery_service               speakers=['RADIO']  first_tokens='Now for the'
miyazaki_my_neighbor_totoro                   speakers=['Satsuki' 'Father']  first_tokens='Father caramel Oh'
miyazaki_nausicaa                             speakers=['YUPA']  first_tokens='Another village dead'
miyazaki_ponyo                                speakers=['LISA']  first_tokens='Sousuke come right'
miyazaki_porco_rosso                          speakers=['Porco Rosso' 'Phone voice']  first_tokens='Yeah Porco Rosso'
miyazaki_princess_mononoke                    speakers=['ASHITAKA' 'KAYA']  first_tokens='Yakkuru Ani-sama Hii-sama'
miyazaki_spirited_away                        speakers=['CHIHIRO']  first_tokens='I ll miss'
miyazaki_the_wind_rises                       speakers=['CAPTION']  first

## Save Tables

In [23]:
LIB.to_csv(f'{OUTPUT_DIR}/LIB.csv')
CORPUS.to_csv(f'{OUTPUT_DIR}/CORPUS.csv')
VOCAB.to_csv(f'{OUTPUT_DIR}/VOCAB.csv')
print(f"Saved:")
print(f"  LIB    — {len(LIB)} films          → {OUTPUT_DIR}/LIB.csv")
print(f"  CORPUS — {len(CORPUS):,} tokens    → {OUTPUT_DIR}/CORPUS.csv")
print(f"  VOCAB  — {len(VOCAB):,} unique terms → {OUTPUT_DIR}/VOCAB.csv")

Saved:
  LIB    — 14 films          → tables/LIB.csv
  CORPUS — 100,506 tokens    → tables/CORPUS.csv
  VOCAB  — 7,367 unique terms → tables/VOCAB.csv
